# **DATA PULL, CLEANING AND VISUALIZATION**

## EDA

### **Data Pull**
This code pulls up confirmed exoplanet data from the NASA Exoplanet Archive's public TAP (Table Access Protocol) API and saves it as a raw CSV for the
DataPrep_EDA tab of my project.

> On githuhb repo - # fetch_exoplanet_data.py

> Also downloaded the data from -

https://phl.upr.edu/hwc/data



In [1]:
# Impoting Libraries.
import pandas as pd
import requests

In [2]:
# TAP STuff - table access protocol (like API request for astronomy)
# what I'm doing after this "requests.get(TAP_URL, params=params)"- This is an HTTP request to a server, with structured parameters,
# returning structured data back (CSV) that the code parses programmatically.
# That's precisely what an API is: a defined way for code to talk to a server and get data back, no browser, no manual download, no clicking through a webpage.
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

In [3]:
# Columns from the Planetary Systems Composite Parameters table
# (pscomppars): one best estimate row per confirmed planet.
COLUMNS = [
    "pl_name",        # planet name
    "hostname",       # host star name
    "discoverymethod",# transit, radial velocity, imaging, microlensing, etc. etc.
    "disc_year",      # year of discovery
    "pl_orbper",      # orbital period (days)
    "pl_rade",        # planet radius (Earth radii)
    "pl_bmasse",      # planet mass (Earth masses)
    "pl_eqt",         # equilibrium temperature (K)
    "st_teff",        # host star effective temperature (K)
    "st_rad",         # host star radius (solar radii)
    "st_mass",        # host star mass (solar masses)
    "sy_dist",        # system distance from Earth (parsecs)
]


In [4]:
def fetch_raw_data() -> pd.DataFrame:
    query = f"select {','.join(COLUMNS)} from pscomppars"
    params = {"query": query, "format": "csv"}
    response = requests.get(TAP_URL, params=params, timeout=60)
    response.raise_for_status()

# requests.Response has .url with the final, fully-encoded GET URL.
# This is the exact request in the DataPrep_EDA section on my site.
    print("Request URL:", response.url)

    from io import StringIO
    return pd.read_csv(StringIO(response.text))

if __name__ == "__main__":
    df = fetch_raw_data()
    df.to_csv("exoplanets_raw.csv", index=False)
    print(f"Saved {len(df)} rows to exoplanets_raw.csv")
    print(df.head())

Request URL: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name%2Chostname%2Cdiscoverymethod%2Cdisc_year%2Cpl_orbper%2Cpl_rade%2Cpl_bmasse%2Cpl_eqt%2Cst_teff%2Cst_rad%2Cst_mass%2Csy_dist+from+pscomppars&format=csv
Saved 6473 rows to exoplanets_raw.csv
         pl_name     hostname discoverymethod  disc_year   pl_orbper  \
0   Kepler-257 d   Kepler-257         Transit       2014   24.664551   
1     TOI-1758 b     TOI-1758         Transit       2024   20.705075   
2   Kepler-607 c   Kepler-607         Transit       2021   39.676800   
3  Kepler-1040 b  Kepler-1040         Transit       2016  201.121400   
4   Kepler-343 b   Kepler-343         Transit       2014    8.968550   

    pl_rade  pl_bmasse   pl_eqt  st_teff  st_rad  st_mass   sy_dist  
0  4.950000      21.70   567.00   5180.0   1.037    0.916  780.2560  
1  3.620000       6.90   551.44   5169.0   0.810    0.880   96.6215  
2  2.760012       8.05   473.00   5244.0   0.888    0.808  519.1220  
3  2.080000   

### **CLEANING AND VISUALIZATION**

 > On githuhb repo - # clean_and_visualize.py

Cleans the raw NASA Exoplanet Archive pull, merges in habitability labels from the PHL Habitable Worlds Catalog and produces the 11 visualizations and two data-preview images used on the Cleaning & Prep and EDA tabs.

Inputs:

    exoplanets_raw.csv   from fetch_exoplanet_data.py

    hwc.csv              the "Full Catalog (CSV)" downloaded from phl.upr.edu/hwc/data

    hwc_table_all.csv    the 70-row shortlist table (Table 1 + Table 2) from the same page, saved as CSV

Outputs:

    exoplanets_clean.csv       cleaned + merged dataset (20 columns)

    charts/01_missingness.png ... charts/12_top_esi_table.png

    charts/raw_preview.png, charts/clean_preview.png

In [5]:
# Importing more libraries -
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [6]:
# the theme to match the site's CSS variables in style.css # Looked at documentations etc. for this one.
BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "grid.color": LINE,
    "font.size": 11, "axes.titlesize": 13, "axes.titlecolor": TEXT,})

os.makedirs("charts", exist_ok=True)

df = pd.read_csv("exoplanets_raw.csv")

In [7]:
# CLEANING
clean = df.copy()
# checking for duplicate planet names
print("Duplicate planet names:", clean["pl_name"].duplicated().sum())

# checking for negative or zero values in the numeric columns, which would be
# physically impossible for these measurements
numeric_cols = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]
for col in numeric_cols:
    n_bad = (clean[col] <= 0).sum()
    print(f"{col}: {n_bad} values <= 0")

# checking how much is missing in each column
print("\nMissing values per column:")
print(clean.isna().sum())

# 157 planets sit above the ~13 Jupiter mass (4131 Earth mass) boundary,
# where "planet" starts overlapping with "brown dwarf" - flagged rather
# than dropped, since they are still confirmed entries in the archive
clean["is_borderline_massive"] = clean["pl_bmasse"] > 4131
print("\nBorderline massive planets flagged:", clean["is_borderline_massive"].sum())

Duplicate planet names: 0
pl_orbper: 0 values <= 0
pl_rade: 0 values <= 0
pl_bmasse: 0 values <= 0
pl_eqt: 0 values <= 0
st_teff: 0 values <= 0
st_rad: 0 values <= 0
st_mass: 0 values <= 0
sy_dist: 0 values <= 0

Missing values per column:
pl_name              0
hostname             0
discoverymethod      0
disc_year            0
pl_orbper          358
pl_rade             50
pl_bmasse           31
pl_eqt             542
st_teff            311
st_rad             336
st_mass              9
sy_dist             28
dtype: int64

Borderline massive planets flagged: 158


- Duplicate planet names and no negative/zero physical values were found in any numeric column on inspection, so nothing needed to be dropped on those grounds.

- Missingness is left as NaN rather than imputed here, since the right way to fill it depends on which model uses that column later.

- Flag planets above the ~13-Jupiter-mass (4131 Earth-mass)boundary, where "planet" starts overlapping with "brown dwarf" by common convention.

- Kept (they are confirmed in the archive) but flagged rather than treated as ordinary planets.
- As for the plots: JUST saving my images and charts in the back instead of plt.show() here.

In [8]:
# Discretize planet size into standard exoplanet-science size classes.
def size_class(r):
    if pd.isna(r): return np.nan
    if r < 1.25: return "Earth-sized"
    if r < 2.0: return "Super-Earth"
    if r < 6.0: return "Sub-Neptune/Neptune"
    return "Giant"
clean["size_class"] = clean["pl_rade"].apply(size_class)

# Discretize equilibrium temperature into a rough habitability zone.
# ~200-320 K is the loose range where liquid water is plausible.
def temp_zone(t):
    if pd.isna(t): return np.nan
    if t < 200: return "Too Cold"
    if t <= 320: return "Temperate"
    return "Too Hot"
clean["temp_zone"] = clean["pl_eqt"].apply(temp_zone)

In [9]:
# MERGE WITH PHL HABITABLE WORLDS CATALOG
hwc = pd.read_csv("hwc.csv")
hwc_cols = hwc[["P_NAME", "P_HABITABLE", "P_ESI", "P_TYPE",
                "P_HABZONE_OPT", "P_HABZONE_CON"]].rename(columns={"P_NAME": "pl_name"})

clean = clean.merge(hwc_cols, on="pl_name", how="left")
clean["hwc_habitable_label"] = clean["P_HABITABLE"].map(
    {0: "Not habitable", 1: "Conservative sample", 2: "Optimistic sample"})

clean.to_csv("exoplanets_clean.csv", index=False)
print("Cleaned + merged file saved:", clean.shape)
print(clean["size_class"].value_counts(dropna=False))
print(clean["temp_zone"].value_counts(dropna=False))
print(clean["hwc_habitable_label"].value_counts(dropna=False))

# PREVIEW TABLE IMAGES
def save_table_image(d, cols, fname, title):
    sample = d[cols].head(6)
    fig, ax = plt.subplots(figsize=(11, 2.2))
    ax.axis("off")
    ax.set_title(title, loc="left", color=GOLD, fontsize=12, pad=14)
    tbl = ax.table(cellText=sample.round(2).astype(str).values,
                    colLabels=cols, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LINE)
        if row == 0:
            cell.set_facecolor(PANEL)
            cell.set_text_props(color=GOLD, weight="bold")
        else:
            cell.set_facecolor(BG)
            cell.set_text_props(color=TEXT)
    plt.savefig(f"charts/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

save_table_image(df, ["pl_name", "hostname", "discoverymethod", "disc_year", "pl_rade", "pl_bmasse"],
                  "raw_preview.png", "Raw data (as pulled from the API)")
save_table_image(clean, ["pl_name", "pl_rade", "size_class", "pl_eqt", "temp_zone", "is_borderline_massive"],
                  "clean_preview.png", "Cleaned data (derived columns added)")

Cleaned + merged file saved: (6473, 21)
size_class
Sub-Neptune/Neptune    2445
Giant                  2180
Super-Earth            1210
Earth-sized             588
NaN                      50
Name: count, dtype: int64
temp_zone
Too Hot      5236
NaN           542
Temperate     364
Too Cold      331
Name: count, dtype: int64
hwc_habitable_label
Not habitable          5495
NaN                     908
Optimistic sample        41
Conservative sample      29
Name: count, dtype: int64


### **VISUALIZATIONS**

Plotting and saving charts in the backend

In [10]:
# VISUALIZATIONS
# JUST saving my images and charts in the back instead of plt.show() here.
# 1. Missingness per column
miss = df.isna().mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(miss.index, miss.values, color=TEAL)
ax.set_xlabel("% missing")
ax.set_title("Missing data by column")
ax.invert_yaxis()
plt.tight_layout(); plt.savefig("charts/01_missingness.png", dpi=150); plt.close()

# 2. Discovery method counts (log scale)
dm = df["discoverymethod"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(dm.index[::-1], dm.values[::-1], color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Number of confirmed planets (log scale)")
ax.set_title("Confirmed planets by discovery method")
plt.tight_layout(); plt.savefig("charts/02_discovery_method.png", dpi=150); plt.close()

# 3. Discoveries per year
yr = df["disc_year"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yr.index, yr.values, color=RUST, linewidth=2)
ax.fill_between(yr.index, yr.values, color=RUST, alpha=0.15)
ax.set_xlabel("Year"); ax.set_ylabel("Planets confirmed")
ax.set_title("Confirmed exoplanet discoveries per year")
plt.tight_layout(); plt.savefig("charts/03_discoveries_per_year.png", dpi=150); plt.close()

# 4. Planet radius distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_rade"].dropna(), bins=40, color=TEAL)
ax.set_xscale("log")
ax.set_xlabel("Planet radius (Earth radii, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet radius")
plt.tight_layout(); plt.savefig("charts/04_radius_distribution.png", dpi=150); plt.close()

# 5. Planet mass distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_bmasse"].dropna(), bins=40, color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Planet mass (Earth masses, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet mass")
plt.tight_layout(); plt.savefig("charts/05_mass_distribution.png", dpi=150); plt.close()

# 6. Mass vs radius scatter
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_rade"], df["pl_bmasse"], s=8, alpha=0.4, color=TEAL)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Radius (Earth radii, log)"); ax.set_ylabel("Mass (Earth masses, log)")
ax.set_title("Planet mass vs. radius")
plt.tight_layout(); plt.savefig("charts/06_mass_vs_radius.png", dpi=150); plt.close()

# 7. Orbital period vs equilibrium temperature
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_orbper"], df["pl_eqt"], s=8, alpha=0.4, color=RUST)
ax.set_xscale("log")
ax.set_xlabel("Orbital period (days, log scale)"); ax.set_ylabel("Equilibrium temperature (K)")
ax.set_title("Orbital period vs. equilibrium temperature")
plt.tight_layout(); plt.savefig("charts/07_period_vs_temp.png", dpi=150); plt.close()

# 8. Host star temperature histogram
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["st_teff"].dropna(), bins=40, color=GOLD)
ax.axvline(5778, color=TEXT, linestyle="--", linewidth=1, label="Sun (5778 K)")
ax.set_xlabel("Host star effective temperature (K)"); ax.set_ylabel("Count")
ax.set_title("Host star temperature")
ax.legend()
plt.tight_layout(); plt.savefig("charts/08_star_temperature.png", dpi=150); plt.close()

# 9. Size class breakdown
sc = clean["size_class"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(sc.index, sc.values, color=[TEAL, GOLD, RUST, DIM])
ax.set_ylabel("Count")
ax.set_title("Planets by size class")
plt.tight_layout(); plt.savefig("charts/09_size_class.png", dpi=150); plt.close()

# 10. Temperature zone breakdown
tz = clean["temp_zone"].value_counts().reindex(["Too Cold", "Temperate", "Too Hot"])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(tz.index, tz.values, color=[TEAL, GOLD, RUST])
ax.set_ylabel("Count")
ax.set_title("Planets by equilibrium-temperature zone")
plt.tight_layout(); plt.savefig("charts/10_temp_zone.png", dpi=150); plt.close()

# 11. Official HWC habitability classification
counts = clean["hwc_habitable_label"].value_counts().reindex(
    ["Not habitable", "Conservative sample", "Optimistic sample"]
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(counts.index, counts.values, color=[DIM, TEAL, GOLD])
ax.set_ylabel("Count")
ax.set_title("Official Habitable Worlds Catalog classification")
for i, v in enumerate(counts.values):
    ax.text(i, v + 40, str(int(v)), ha="center", color=TEXT, fontsize=10)
plt.tight_layout(); plt.savefig("charts/11_hwc_habitability.png", dpi=150); plt.close()

# 12. Top 10 most Earth-like worlds by ESI, from the shortlist table
short = pd.read_csv("hwc_table_all.csv")
short.columns = [re.sub(r"<[^>]+>", "", c) for c in short.columns]
short = short.sort_values("ESI", ascending=False).head(10)
short_display = short[["Name", "Type", "ESI"]].round({"ESI": 3})

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.axis("off")
ax.set_title("Top 10 potentially habitable worlds, by Earth Similarity Index",
             loc="left", color=GOLD, fontsize=12, pad=14)
tbl = ax.table(cellText=short_display.values, colLabels=["Name", "Type", "ESI"],
                cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(LINE)
    if row == 0:
        cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
    else:
        cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
plt.savefig("charts/12_top_esi_table.png", dpi=150, bbox_inches="tight")
plt.close()

print("All charts saved to charts/")

All charts saved to charts/


# **MODULE 2**
## Clustering and PCA

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

In [12]:
os.makedirs("charts", exist_ok=True)

df = pd.read_csv("exoplanets_clean.csv")

# DATA PREP
# Clustering and PCA need unlabeled numeric data only, so just the eight
# physical measurement columns are used here, with rows missing any of
# them dropped.
numeric_cols = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt",
                 "st_teff", "st_rad", "st_mass", "sy_dist"]
cluster_df = df.dropna(subset=numeric_cols).reset_index(drop=True)
print("Rows with complete numeric data:", len(cluster_df), "of", len(df))

# these columns span several orders of magnitude (orbital period especially),
# so log-transform first so no single feature dominates the distance purely
# because of its raw scale, then standardize to mean 0 / std 1
X_log = np.log10(cluster_df[numeric_cols])
X_log.columns = [c + "_log" for c in numeric_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

#  K-MEANS
silhouette_by_k = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_by_k[k] = silhouette_score(X_scaled, labels)
    print(f"k={k}: silhouette={silhouette_by_k[k]:.4f}")

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print("Best k by silhouette:", best_k)

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = km_best.fit_predict(X_scaled)

# silhouette score vs k
ks = sorted(silhouette_by_k.keys())
vals = [silhouette_by_k[k] for k in ks]
plt.figure(figsize=(7, 4.5))
plt.plot(ks, vals, marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("Average silhouette score")
plt.title("K-means silhouette score by k")
plt.xticks(ks)
plt.tight_layout()
plt.savefig("charts/silhouette_by_k.png", dpi=150)
plt.close()

# PCA (computed early so clusters can be visualized in PC space)
pca = PCA()
pcs = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
print("Explained variance ratio:", evr)
print("Cumulative (first 2):", evr[:2].sum())

loadings = pd.DataFrame(pca.components_[:2].T, index=X_log.columns, columns=["PC1", "PC2"])
print(loadings.round(3))

# k-means clusters shown in PC1/PC2 space
plt.figure(figsize=(7, 6))
scatter = plt.scatter(pcs[:, 0], pcs[:, 1], c=kmeans_labels, s=8, alpha=0.5, cmap="tab10")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"K-means clusters (k={best_k}), shown in PC1/PC2 space")
plt.legend(*scatter.legend_elements(), title="Cluster", fontsize=8)
plt.tight_layout()
plt.savefig("charts/kmeans_clusters_pca.png", dpi=150)
plt.close()

# HIERARCHICAL CLUSTERING (cosine distance)
# cosine distance can't be combined with ward/centroid linkage, so average
# linkage is used instead
dist = pdist(X_scaled, metric="cosine")
Z = linkage(dist, method="average")

plt.figure(figsize=(9, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8, show_contracted=True)
plt.xlabel("Cluster (count of original points in parentheses)")
plt.ylabel("Cosine distance")
plt.title("Hierarchical clustering dendrogram (average linkage, cosine distance)")
plt.tight_layout()
plt.savefig("charts/dendrogram.png", dpi=150)
plt.close()

# reading the dendrogram's own suggested k from the biggest jump in merge height
# among the last several merges
last_merges = Z[-10:, 2]
diffs = np.diff(last_merges)
jump_idx = np.argmax(diffs)
suggested_k = len(last_merges) - jump_idx
print("Dendrogram-suggested k (largest merge-height jump):", suggested_k)

# comparing k-means and hierarchical clustering at the same k
hclust_labels = fcluster(Z, t=best_k, criterion="maxclust")
ari = adjusted_rand_score(kmeans_labels, hclust_labels)
print(f"Adjusted Rand Index, k-means vs hierarchical at k={best_k}:", round(ari, 4))

contingency = pd.crosstab(pd.Series(kmeans_labels, name="kmeans"),
                           pd.Series(hclust_labels, name="hclust"))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(contingency.values, cmap="Blues")
ax.set_xticks(range(len(contingency.columns))); ax.set_xticklabels(contingency.columns)
ax.set_yticks(range(len(contingency.index))); ax.set_yticklabels(contingency.index)
ax.set_xlabel("Hierarchical cluster"); ax.set_ylabel("K-means cluster")
ax.set_title("How K-means and hierarchical clusters overlap")
for i in range(contingency.shape[0]):
    for j in range(contingency.shape[1]):
        ax.text(j, i, contingency.values[i, j], ha="center", va="center", fontsize=9)
plt.colorbar(im, label="Number of planets")
plt.tight_layout()
plt.savefig("charts/kmeans_vs_hclust.png", dpi=150)
plt.close()

# PCA VISUALIZATIONS
plt.figure(figsize=(7, 4.5))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="black", label="Cumulative")
plt.xlabel("Principal component"); plt.ylabel("Proportion of variance explained")
plt.title("PCA scree plot")
plt.legend()
plt.tight_layout()
plt.savefig("charts/pca_scree.png", dpi=150)
plt.close()

plt.figure(figsize=(7, 6))
plt.scatter(pcs[:, 0], pcs[:, 1], s=6, alpha=0.3)
plt.xlabel(f"PC1 ({evr[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({evr[1]*100:.1f}% variance)")
plt.title("Exoplanets projected onto the first two principal components")
plt.tight_layout()
plt.savefig("charts/pca_pc1_pc2.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pcs[:, 0], pcs[:, 1], s=5, alpha=0.15, color="gray")
for feat in loadings.index:
    x, y = loadings.loc[feat, "PC1"] * 6, loadings.loc[feat, "PC2"] * 6
    ax.arrow(0, 0, x, y, color="red", width=0.02, head_width=0.15)
    ax.text(x * 1.1, y * 1.1, feat.replace("_log", ""), color="red", fontsize=9, ha="center")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("PCA loadings: which features drive PC1 and PC2")
plt.tight_layout()
plt.savefig("charts/pca_loadings.png", dpi=150)
plt.close()

print("Module 2 charts saved to charts/")

Rows with complete numeric data: 5775 of 6473
k=2: silhouette=0.3280
k=3: silhouette=0.3621
k=4: silhouette=0.3482
k=5: silhouette=0.3629
k=6: silhouette=0.3189
k=7: silhouette=0.2672
Best k by silhouette: 5
Explained variance ratio: [0.40054941 0.3213914  0.12498729 0.0703799  0.05592775 0.01283822
 0.00991447 0.00401155]
Cumulative (first 2): 0.721940811341963
                 PC1    PC2
pl_orbper_log  0.270 -0.435
pl_rade_log    0.421 -0.249
pl_bmasse_log  0.423 -0.284
pl_eqt_log     0.016  0.532
st_teff_log    0.357  0.318
st_rad_log     0.456  0.113
st_mass_log    0.473  0.255
sy_dist_log    0.106  0.455
Dendrogram-suggested k (largest merge-height jump): 2
Adjusted Rand Index, k-means vs hierarchical at k=5: 0.5836
Module 2 charts saved to charts/


## Module 3 data prep: building the labeled dataset and split it into training and testing sets.
#### Input: exoplanets_clean.csv (from clean_and_visualize.py)
#### Output: labeled_sample.csv, train_set.csv, test_set.csv, plus small preview images in charts/

In [13]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# keeping the dark bg going :)
BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 13, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
df = pd.read_csv("exoplanets_clean.csv")

features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# supervised methods need a label. The label here comes from the PHL Habitable Worlds
# Catalog: P_HABITABLE is 0 (not habitable), 1 (conservative sample) or 2 (optimistic sample).
# Collapsed into one yes/no label: habitable = 1 if the catalog lists the planet in either sample.
# Only planets the catalog has assessed (P_HABITABLE present) and with all 8 measurements are kept.
labeled = df.dropna(subset=features + ["P_HABITABLE"]).copy()
labeled["habitable"] = (labeled["P_HABITABLE"] > 0).astype(int)
labeled = labeled[["pl_name"] + features + ["habitable"]].reset_index(drop=True)
print("Labeled rows:", len(labeled))
print(labeled["habitable"].value_counts())

# 70/30 split. stratify keeps the same share of habitable planets in both sets,
# which matters because there are so few of them. random_state makes it repeatable.
train, test = train_test_split(labeled, test_size=0.30, random_state=42, stratify=labeled["habitable"])

# the two sets must not share a single planet, otherwise the test score would be meaningless
overlap = set(train["pl_name"]) & set(test["pl_name"])
print("Planets in both sets:", len(overlap))
assert len(overlap) == 0

labeled.to_csv("labeled_sample.csv", index=False)
train.to_csv("train_set.csv", index=False)
test.to_csv("test_set.csv", index=False)
print("Train:", len(train), "habitable:", train["habitable"].sum())
print("Test:", len(test), "habitable:", test["habitable"].sum())

def save_table_image(d, fname, title):
    sample = d.head(6).copy()
    for c in features:
        sample[c] = sample[c].map(lambda v: f"{v:.3g}")
    fig, ax = plt.subplots(figsize=(11, 2.3))
    ax.axis("off")
    ax.set_title(title, loc="left", color=GOLD, fontsize=12, pad=14)
    tbl = ax.table(cellText=sample.values, colLabels=sample.columns, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.5)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LINE)
        if row == 0:
            cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
        else:
            cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
    plt.savefig(f"charts/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

save_table_image(labeled, "labeled_preview.png", f"Labeled sample ({len(labeled)} planets, {labeled['habitable'].sum()} habitable)")
save_table_image(train, "train_preview.png", f"Training set ({len(train)} planets, {train['habitable'].sum()} habitable)")
save_table_image(test, "test_preview.png", f"Testing set ({len(test)} planets, {test['habitable'].sum()} habitable)")
print("done")

Labeled rows: 5048
habitable
0    4982
1      66
Name: count, dtype: int64
Planets in both sets: 0
Train: 3533 habitable: 46
Test: 1515 habitable: 20
done


## Module 3: Naive Bayes (multinomial) on the labeled exoplanet data.
#### Input:  train_set.csv, test_set.csv (from prep_train_test.py)
#### Output: confusion matrix chart in charts/


In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")
features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# multinomial NB treats each feature like a count, so every value has to be zero or
# positive. The measurements span several orders of magnitude, so take log10 first,
# then squeeze each column into 0 to 1. The scaler is fit on the training set only,
# and test values that fall outside the training range are clipped to 0 or 1.
X_train = np.log10(train[features])
X_test = np.log10(test[features])
scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = np.clip(scaler.transform(X_test), 0, 1)
y_train, y_test = train["habitable"], test["habitable"]

def report(name, model):
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    print(f"\n{name}")
    print("accuracy:", round(acc, 4), "precision:", round(prec, 4), "recall:", round(rec, 4))
    print(cm)
    return cm, acc

print("Baseline (always guess 'not habitable') accuracy:", round(1 - y_test.mean(), 4))

# model 1: default, the class prior is learned from the training set
nb_default = MultinomialNB().fit(X_train, y_train)
cm1, acc1 = report("MultinomialNB, learned prior", nb_default)

# model 2: same model but both classes get an equal prior
nb_equal = MultinomialNB(fit_prior=False).fit(X_train, y_train)
cm2, acc2 = report("MultinomialNB, equal prior", nb_equal)

cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, cm, acc, title in [(axes[0], cm1, acc1, "Learned prior"), (axes[1], cm2, acc2, "Equal prior")]:
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not habitable", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not habitable", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{title} (accuracy {acc*100:.1f}%)")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout()
plt.savefig("charts/nb_confusion_matrices.png", dpi=150)
plt.close()
print("saved charts/nb_confusion_matrices.png")

Baseline (always guess 'not habitable') accuracy: 0.9868

MultinomialNB, learned prior
accuracy: 0.9868 precision: 0.0 recall: 0.0
[[1495    0]
 [  20    0]]

MultinomialNB, equal prior
accuracy: 0.8224 precision: 0.0692 recall: 1.0
[[1226  269]
 [   0   20]]
saved charts/nb_confusion_matrices.png


## Module 3: Decision Trees on the labeled exoplanet data.
#### Input:  train_set.csv, test_set.csv (from prep_train_test.py)
#### Output: three tree diagrams, confusion matrices, and a feature importance chart in charts/


In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")
features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# trees split on thresholds, so no scaling or log transform is needed
X_train, y_train = train[features], train["habitable"]
X_test, y_test = test[features], test["habitable"]

print("Baseline (always guess 'not habitable') accuracy:", round(1 - y_test.mean(), 4))

# three different trees: two split rules (gini, entropy) and a third that weights the rare class more
specs = [
    ("Tree 1: Gini, max depth 3", DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)),
    ("Tree 2: Entropy, max depth 3", DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)),
    ("Tree 3: Gini, max depth 4, balanced class weights",
     DecisionTreeClassifier(criterion="gini", max_depth=4, class_weight="balanced", random_state=42)),]

def theme_tree(artists):
    # boxes: dark panel; leaves that predict "habitable" are outlined in gold.
    # plot_tree also returns the small True/False edge labels, which have no box,
    # so each artist is judged by its own text instead of its position in the list.
    for a in artists:
        a.set_color(TEXT)
        if a.arrow_patch is not None:
            a.arrow_patch.set_color(DIM)
        patch = a.get_bbox_patch()
        if patch is None:
            continue
        txt = a.get_text()
        is_leaf = "<=" not in txt.split("\n")[0]
        predicts_hab = "class = habitable" in txt
        patch.set_facecolor(PANEL)
        patch.set_edgecolor(GOLD if (is_leaf and predicts_hab) else LINE)
        patch.set_linewidth(2 if (is_leaf and predicts_hab) else 1)

results = []
for i, (name, model) in enumerate(specs, start=1):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    results.append((name, cm, acc))
    print(f"\n{name}")
    print("accuracy:", round(acc, 4), "precision:", round(prec, 4), "recall:", round(rec, 4))
    print(cm)

    fig, ax = plt.subplots(figsize=(14, 7))
    artists = plot_tree(model, feature_names=features, class_names=["not habitable", "habitable"],
                        filled=False, rounded=True, fontsize=8, ax=ax, impurity=True)
    theme_tree(artists)
    ax.set_title(name)
    plt.tight_layout()
    plt.savefig(f"charts/dt_tree_{i}.png", dpi=150)
    plt.close()

# confusion matrices for all three trees side by side
cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (name, cm, acc) in zip(axes, results):
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not hab.", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not hab.", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{name.split(':')[0]} (accuracy {acc*100:.1f}%)", fontsize=11)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout()
plt.savefig("charts/dt_confusion_matrices.png", dpi=150)
plt.close()

# which measurements the first tree actually used
imp = pd.Series(specs[0][1].feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(imp.index, imp.values, color=TEAL)
ax.set_xlabel("Feature importance")
ax.set_title("Which measurements Tree 1 relies on")
plt.tight_layout()
plt.savefig("charts/dt_feature_importance.png", dpi=150)
plt.close()
print("\nsaved charts")

Baseline (always guess 'not habitable') accuracy: 0.9868

Tree 1: Gini, max depth 3
accuracy: 0.9914 precision: 0.7059 recall: 0.6
[[1490    5]
 [   8   12]]

Tree 2: Entropy, max depth 3
accuracy: 0.9901 precision: 0.619 recall: 0.65
[[1487    8]
 [   7   13]]

Tree 3: Gini, max depth 4, balanced class weights
accuracy: 0.9842 precision: 0.4524 recall: 0.95
[[1472   23]
 [   1   19]]

saved charts


### MODULE 4 - REGRESSIONS and Boosting
#### 1. LINEAR AND LOGISTIC Regressions and how they compare to NB (NAIVE BAYES)

In [16]:
# Module 4: linear regression and logistic regression.
# Input:train_set.csv, test_set.csv (from prep_train_test.py)
# Output:regression charts in charts/
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, confusion_matrix, precision_score, recall_score)
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT,
    "legend.facecolor": PANEL, "legend.edgecolor": LINE})

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")

In [17]:
# LINEAR REGRESSION
# target: planet mass (log10 Earth masses), a number.
# Predictors: the other 7 measurements, also log10, since these span several orders of magnitude and the mass-radius relationship is close to a straight line on log-log axes.
predictors = ["pl_orbper", "pl_rade", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# Many planets in the archive's composite table have one of mass or radius filled in from the other with a published piecewise mass-radius formula (Chen and Kipping style:
# R = 1.008 M^0.279 below 2.04 Earth masses, 0.808 M^0.589 up to 131.6, 17.74 M^-0.044 above).
# A planet within 1% of that curve is flagged, because for it the mass is close to a formula of the radius and not an independent measurement.
def on_formula_curve(d):
    M = d["pl_bmasse"]
    r_formula = np.where(M < 2.04, 1.008 * M ** 0.279, np.where(M < 131.6, 0.808 * M ** 0.589, 17.74 * M ** -0.044))
    return (np.abs(d["pl_rade"] - r_formula) / d["pl_rade"] < 0.01).to_numpy()

train["on_curve"] = on_formula_curve(train)
test["on_curve"] = on_formula_curve(test)
print("Within 1% of the formula curve: training", int(train["on_curve"].sum()), "of", len(train),
      "| testing", int(test["on_curve"].sum()), "of", len(test))

def lin_scores(y_true, y_pred):
    return (mean_squared_error(y_true, y_pred) ** 0.5, mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred))

X_train = np.log10(train[predictors]); X_test = np.log10(test[predictors])
y_train = np.log10(train["pl_bmasse"]); y_test = np.log10(test["pl_bmasse"])

# model A: all training planets
lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)
rmse, mae, r2 = lin_scores(y_test, pred)
base_rmse = mean_squared_error(y_test, np.full(len(y_test), y_train.mean())) ** 0.5
print("\nLinear regression, all planets (log10 mass)")
print("RMSE:", round(rmse, 3), "MAE:", round(mae, 3), "R2:", round(r2, 3), "| RMSE of guessing the training mean:", round(base_rmse, 3))
print("typical error as a factor: RMSE", round(10 ** rmse, 2), "x, MAE", round(10 ** mae, 2), "x")
oc = test["on_curve"].to_numpy()
r_on = lin_scores(y_test[oc], pred[oc]); r_off = lin_scores(y_test[~oc], pred[~oc])
print("  model A on formula-curve planets: RMSE %.3f MAE %.3f R2 %.3f" % r_on)
print("  model A on other planets:         RMSE %.3f MAE %.3f R2 %.3f" % r_off)
print("intercept:", round(float(lin.intercept_), 3))
for name, c in zip(predictors, lin.coef_):
    print(f"  {name}: {c:.3f}  (doubling it multiplies predicted mass by {2 ** c:.2f})")

# model B: trained and tested only on planets that are NOT on the formula curve
tr_m, te_m = train[~train["on_curve"]], test[~test["on_curve"]]
lin_m = LinearRegression().fit(np.log10(tr_m[predictors]), np.log10(tr_m["pl_bmasse"]))
pred_m = lin_m.predict(np.log10(te_m[predictors])); y_m = np.log10(te_m["pl_bmasse"])
rmse_m, mae_m, r2_m = lin_scores(y_m, pred_m)
base_m = mean_squared_error(y_m, np.full(len(y_m), np.log10(tr_m["pl_bmasse"]).mean())) ** 0.5
print("\nLinear regression, planets off the formula curve only: train", len(tr_m), "test", len(te_m))
print("RMSE:", round(rmse_m, 3), "MAE:", round(mae_m, 3), "R2:", round(r2_m, 3), "| baseline RMSE:", round(base_m, 3))
print("typical error as a factor: RMSE", round(10 ** rmse_m, 2), "x, MAE", round(10 ** mae_m, 2), "x")
for name, c in zip(predictors, lin_m.coef_):
    print(f"  {name}: {c:.3f}  (doubling it multiplies predicted mass by {2 ** c:.2f})")

# predicted vs actual, colored by whether the planet sits on the formula curve
lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
plt.figure(figsize=(6.8, 6))
plt.scatter(y_test[oc], pred[oc], s=10, alpha=0.4, color=TEAL, label=f"on the formula curve ({oc.sum()})")
plt.scatter(y_test[~oc], pred[~oc], s=12, alpha=0.6, color=GOLD, label=f"off the curve ({(~oc).sum()})")
plt.plot([lo, hi], [lo, hi], color=TEXT, linewidth=1.2, linestyle="--", label="perfect prediction")
plt.xlabel("Actual mass (log10 Earth masses)")
plt.ylabel("Predicted mass (log10 Earth masses)")
plt.title(f"Linear regression: predicted vs actual (R$^2$ = {r2:.2f})")
plt.legend(fontsize=9)
plt.tight_layout(); plt.savefig("charts/lin_pred_vs_actual.png", dpi=150); plt.close()

# coefficients
order = np.argsort(lin.coef_)
plt.figure(figsize=(7, 4.5))
plt.barh(np.array(predictors)[order], lin.coef_[order],
         color=[TEAL if c > 0 else RUST for c in lin.coef_[order]])
plt.axvline(0, color=DIM, linewidth=0.8)
plt.xlabel("Coefficient (change in log10 mass per 1 change in log10 predictor)")
plt.title("Linear regression coefficients")
plt.tight_layout(); plt.savefig("charts/lin_coefficients.png", dpi=150); plt.close()

Within 1% of the formula curve: training 2583 of 3533 | testing 1094 of 1515

Linear regression, all planets (log10 mass)
RMSE: 0.385 MAE: 0.238 R2: 0.858 | RMSE of guessing the training mean: 1.023
typical error as a factor: RMSE 2.43 x, MAE 1.73 x
  model A on formula-curve planets: RMSE 0.276 MAE 0.180 R2 0.926
  model A on other planets:         RMSE 0.580 MAE 0.389 R2 0.617
intercept: 0.953
  pl_orbper: 0.217  (doubling it multiplies predicted mass by 1.16)
  pl_rade: 2.097  (doubling it multiplies predicted mass by 4.28)
  pl_eqt: 0.339  (doubling it multiplies predicted mass by 1.26)
  st_teff: -0.506  (doubling it multiplies predicted mass by 0.70)
  st_rad: 0.114  (doubling it multiplies predicted mass by 1.08)
  st_mass: 0.030  (doubling it multiplies predicted mass by 1.02)
  sy_dist: -0.069  (doubling it multiplies predicted mass by 0.95)

Linear regression, planets off the formula curve only: train 950 test 421
RMSE: 0.526 MAE: 0.381 R2: 0.684 | baseline RMSE: 0.936
typica

In [18]:
# LOGISTIC REGRESSION
# label: habitable (0 or 1). Predictors: all eight measurements, log10 then standardized (scaler fit on the training set only) so coefficients are comparable.
features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]
scaler = StandardScaler().fit(np.log10(train[features]))
A_train = scaler.transform(np.log10(train[features]))
A_test = scaler.transform(np.log10(test[features]))
yb_train, yb_test = train["habitable"], test["habitable"]
print("\nBaseline (always guess 'not habitable') accuracy:", round(1 - yb_test.mean(), 4))

def scores(pred):
    return (accuracy_score(yb_test, pred), recall_score(yb_test, pred, zero_division=0),
            precision_score(yb_test, pred, zero_division=0))

logit = LogisticRegression(max_iter=2000).fit(A_train, yb_train)
logit_bal = LogisticRegression(max_iter=2000, class_weight="balanced").fit(A_train, yb_train)
results = {}
for name, model in [("Logistic, default", logit), ("Logistic, balanced weights", logit_bal)]:
    p = model.predict(A_test)
    results[name] = (confusion_matrix(yb_test, p, labels=[0, 1]), scores(p))
    a, r, pr = results[name][1]
    print(f"\n{name}: accuracy {a:.4f}, recall {r:.3f}, precision {pr:.3f}")
    print(results[name][0])
print("\nLogistic regression coefficients (log-odds per 1 standard deviation), default model:")
for name, c in zip(features, logit.coef_[0]):
    print(f"  {name}: {c:.2f}  (odds multiply by {np.exp(c):.2f})")

cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, (cm, sc)) in zip(axes, results.items()):
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not habitable", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not habitable", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{name} (accuracy {sc[0]*100:.1f}%)")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout(); plt.savefig("charts/logit_confusion_matrices.png", dpi=150); plt.close()

order = np.argsort(logit.coef_[0])
plt.figure(figsize=(7, 4.5))
plt.barh(np.array(features)[order], logit.coef_[0][order],
         color=[TEAL if c > 0 else RUST for c in logit.coef_[0][order]])
plt.axvline(0, color=DIM, linewidth=0.8)
plt.xlabel("Coefficient (change in log-odds per 1 standard deviation)")
plt.title("Logistic regression coefficients (default model)")
plt.tight_layout(); plt.savefig("charts/logit_coefficients.png", dpi=150); plt.close()


Baseline (always guess 'not habitable') accuracy: 0.9868

Logistic, default: accuracy 0.9888, recall 0.300, precision 0.667
[[1492    3]
 [  14    6]]

Logistic, balanced weights: accuracy 0.9406, recall 1.000, precision 0.182
[[1405   90]
 [   0   20]]

Logistic regression coefficients (log-odds per 1 standard deviation), default model:
  pl_orbper: 1.15  (odds multiply by 3.15)
  pl_rade: -1.83  (odds multiply by 0.16)
  pl_bmasse: -1.17  (odds multiply by 0.31)
  pl_eqt: -1.71  (odds multiply by 0.18)
  st_teff: -0.99  (odds multiply by 0.37)
  st_rad: 0.23  (odds multiply by 1.25)
  st_mass: -0.08  (odds multiply by 0.92)
  sy_dist: 0.31  (odds multiply by 1.37)


In [19]:
# COMPARISON WITH NAIVE BAYES (Part 3)
# same gig as naive_bayes.py: log10, scale to 0-1, equal prior
Xn_train = np.log10(train[features]); Xn_test = np.log10(test[features])
mm = MinMaxScaler().fit(Xn_train)
nb = MultinomialNB(fit_prior=False).fit(mm.transform(Xn_train), yb_train)
nb_pred = nb.predict(np.clip(mm.transform(Xn_test), 0, 1))
nb_cm = confusion_matrix(yb_test, nb_pred, labels=[0, 1])
nb_scores = scores(nb_pred)
print("\nNaive Bayes (equal prior):", [round(float(s), 3) for s in nb_scores], nb_cm.tolist())

models = [("Naive Bayes,\nequal prior", nb_scores), ("Logistic,\ndefault", results["Logistic, default"][1]),
          ("Logistic,\nbalanced", results["Logistic, balanced weights"][1])]
labels = ["Accuracy", "Recall", "Precision"]
x = np.arange(len(labels)); w = 0.26
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for k, (mname, sc) in enumerate(models):
    bars = ax.bar(x + (k - 1) * w, sc, w, label=mname, color=[DIM, TEAL, GOLD][k])
    for b, v in zip(bars, sc):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}", ha="center", fontsize=8, color=TEXT)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.12)
ax.set_title("Logistic regression vs Naive Bayes on the same 1,515 testing planets")
ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout(); plt.savefig("charts/logit_vs_nb.png", dpi=150); plt.close()
print("saved charts")


Naive Bayes (equal prior): [0.822, 1.0, 0.069] [[1226, 269], [0, 20]]
saved charts


###  Module 4: REGRESSIONS AND Boosting
#### 2. AdaBoost on the labeled exoplanet data.

In [21]:
import warnings
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT,
    "legend.facecolor": PANEL, "legend.edgecolor": LINE})

# scikit-learn prints a harmless feature-name warning from inside AdaBoost, hidden to keep the output readable
warnings.filterwarnings("ignore", message="X does not have valid feature names")

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")
clean = pd.read_csv("exoplanets_clean.csv")
numeric = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# ONE-HOT ENCODING
# scikit-learn's AdaBoost needs numbers only. The eight measurements are already numeric, but how a planet was discovered (transit, radial velocity, ...) is a category. Each method becomes its own 0/1 column.
# Columns are built from the training set, and the testing set is lined up to the same columns, so a method that only appears in the testing set is ignored.
method = clean.set_index("pl_name")["discoverymethod"]
train["discoverymethod"] = train["pl_name"].map(method)
test["discoverymethod"] = test["pl_name"].map(method)
dummies_train = pd.get_dummies(train["discoverymethod"], prefix="method").astype(int)
dummies_test = pd.get_dummies(test["discoverymethod"], prefix="method").astype(int)
dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
print("Discovery methods in the training set:")
print(train["discoverymethod"].value_counts())

X_train = pd.concat([train[numeric], dummies_train], axis=1)
X_test = pd.concat([test[numeric], dummies_test], axis=1)
y_train, y_test = train["habitable"], test["habitable"]
print("Predictor columns:", X_train.shape[1], "(8 measurements +", dummies_train.shape[1], "one-hot columns)")

# small preview image of the encoding step: one example planet per discovery method
show = train.groupby("discoverymethod").head(1).head(6)
cols = ["pl_name", "discoverymethod"] + [c for c in ["method_Transit", "method_Radial Velocity",
                                                      "method_Microlensing", "method_Imaging"] if c in dummies_train.columns]
preview = pd.concat([show[["pl_name", "discoverymethod"]], dummies_train.loc[show.index]], axis=1)[cols]
fig, ax = plt.subplots(figsize=(11, 2.3))
ax.axis("off")
ax.set_title("One-hot encoding: discovery method becomes 0/1 columns", loc="left", color=GOLD, fontsize=12, pad=14)
tbl = ax.table(cellText=preview.astype(str).values, colLabels=preview.columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(LINE)
    if row == 0:
        cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
    else:
        cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
plt.savefig("charts/ada_onehot_preview.png", dpi=150, bbox_inches="tight")
plt.close()

def make_model(n, lr):
    # a decision stump is a tree with a single split (max_depth=1)
    return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                              n_estimators=n, learning_rate=lr, random_state=42)

# TUNING
# the testing set is not touched here. Rounds and learning rate are chosen with 5-fold cross-validation inside the training set: each fold is held out as validation data once.
# staged_predict gives the prediction after every round, so one fit covers all round counts.
rounds = 200
learning_rates = [0.01, 0.1, 0.5, 1.0]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_curves = {lr: np.zeros(rounds) for lr in learning_rates}
f1_curves = {lr: np.zeros(rounds) for lr in learning_rates}
for lr in learning_rates:
    for fit_idx, val_idx in cv.split(X_train, y_train):
        m = make_model(rounds, lr).fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
        for r, p in enumerate(m.staged_predict(X_train.iloc[val_idx])):
            acc_curves[lr][r] += accuracy_score(y_train.iloc[val_idx], p) / 5
            f1_curves[lr][r] += f1_score(y_train.iloc[val_idx], p, zero_division=0) / 5

# accuracy barely moves (98.7% of planets are not habitable), so the choice uses F1 on the habitable class, which does react to finding the rare planets
best_lr, best_n, best_f1 = None, None, -1
for lr in learning_rates:
    r = int(np.argmax(f1_curves[lr]))
    if f1_curves[lr][r] > best_f1:
        best_lr, best_n, best_f1 = lr, r + 1, f1_curves[lr][r]
majority = 1 - y_train.mean()
print("\nBest by validation F1: learning rate", best_lr, "rounds", best_n, "F1", round(best_f1, 3))
print("validation accuracy at that setting:", round(acc_curves[best_lr][best_n - 1], 4), "(always-no accuracy:", round(majority, 4), ")")

palette = [DIM, TEAL, GOLD, RUST]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for lr, col in zip(learning_rates, palette):
    axes[0].plot(range(1, rounds + 1), acc_curves[lr], color=col, label=f"learning rate {lr}")
    axes[1].plot(range(1, rounds + 1), f1_curves[lr], color=col, label=f"learning rate {lr}")
axes[0].axhline(majority, color=TEXT, linestyle="--", linewidth=0.8, label="always guess not habitable")
axes[0].set_xlabel("Boosting rounds"); axes[0].set_ylabel("Validation accuracy")
axes[0].set_title("Validation accuracy by round"); axes[0].legend(fontsize=8, loc="upper left")
axes[1].set_xlabel("Boosting rounds"); axes[1].set_ylabel("Validation F1 (habitable class)")
axes[1].set_title("Validation F1 by round")
plt.tight_layout(); plt.savefig("charts/ada_tuning.png", dpi=150); plt.close()

# FINAL MODELS ON THE TESTING SET
def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    p = model.predict(Xte)
    acc = accuracy_score(y_test, p); rec = recall_score(y_test, p, zero_division=0)
    prec = precision_score(y_test, p, zero_division=0); cm = confusion_matrix(y_test, p, labels=[0, 1])
    print(f"\n{name}: accuracy {acc:.4f}, recall {rec:.3f}, precision {prec:.3f}")
    print(cm.tolist())
    return model, cm, (acc, rec, prec)

print("\nBaseline (always guess 'not habitable') accuracy:", round(1 - y_test.mean(), 4))
untuned, cm_untuned, sc_untuned = evaluate("AdaBoost, default settings (50 rounds, learning rate 1.0)", make_model(50, 1.0), X_train, X_test)
final, cm_final, sc_final = evaluate(f"AdaBoost, tuned ({best_n} rounds, learning rate {best_lr})", make_model(best_n, best_lr), X_train, X_test)
_, cm_num, sc_num = evaluate("AdaBoost, tuned settings, measurements only (no one-hot columns)",
                              make_model(best_n, best_lr), X_train[numeric], X_test[numeric])
# the Part 3 decision tree (Tree 1: Gini, depth 3) on the same split
tree = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
_, cm_tree, sc_tree = evaluate("Decision tree from Part 3 (Tree 1: Gini, depth 3)", tree, X_train[numeric], X_test[numeric])

cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (title, cm, sc) in zip(axes, [("AdaBoost, default settings", cm_untuned, sc_untuned),
                                       (f"AdaBoost, tuned ({best_n} rounds)", cm_final, sc_final)]):
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not habitable", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not habitable", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{title} (accuracy {sc[0]*100:.1f}%)", fontsize=11)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout(); plt.savefig("charts/ada_confusion_matrices.png", dpi=150); plt.close()

# alpha (the weight of each stump's vote) for the first rounds of the tuned model
alphas = final.estimator_weights_[:30]
errors = final.estimator_errors_[:30]
print("\nFirst five stump errors:", np.round(errors[:5], 3).tolist())
print("First five alphas:", np.round(alphas[:5], 3).tolist())
print("check, learning rate * ln((1-error)/error) for the first stump:",
      round(best_lr * np.log((1 - errors[0]) / errors[0]), 3))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(1, len(alphas) + 1), alphas, color=TEAL)
ax.set_xlabel("Boosting round"); ax.set_ylabel("Alpha (weight of the stump's vote)")
ax.set_title("Alpha of each stump in the tuned model (first 30 rounds)")
plt.tight_layout(); plt.savefig("charts/ada_alpha_by_round.png", dpi=150); plt.close()

# which predictors the tuned model uses most
imp = pd.Series(final.feature_importances_, index=X_train.columns).sort_values().tail(8)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(imp.index, imp.values, color=TEAL)
ax.set_xlabel("Feature importance")
ax.set_title("What the tuned AdaBoost model relies on")
plt.tight_layout(); plt.savefig("charts/ada_feature_importance.png", dpi=150); plt.close()
print("\nTop importances:", imp.round(3).to_dict())

# comparison with the Part 3 tree
labels = ["Accuracy", "Recall", "Precision"]
x = np.arange(3); w = 0.26
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for k, (mname, sc) in enumerate([("Decision tree (Part 3)", sc_tree), ("AdaBoost, default", sc_untuned), (f"AdaBoost, tuned", sc_final)]):
    bars = ax.bar(x + (k - 1) * w, sc, w, label=mname, color=[DIM, TEAL, GOLD][k])
    for b, v in zip(bars, sc):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}", ha="center", fontsize=8, color=TEXT)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.12)
ax.set_title("AdaBoost vs the Part 3 decision tree on the same 1,515 testing planets")
ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout(); plt.savefig("charts/ada_vs_tree.png", dpi=150); plt.close()
print("saved charts")

Discovery methods in the training set:
discoverymethod
Transit                          2815
Radial Velocity                   696
Imaging                             7
Transit Timing Variations           6
Eclipse Timing Variations           4
Orbital Brightness Modulation       3
Pulsation Timing Variations         1
Astrometry                          1
Name: count, dtype: int64
Predictor columns: 16 (8 measurements + 8 one-hot columns)

Best by validation F1: learning rate 1.0 rounds 58 F1 0.804
validation accuracy at that setting: 0.9946 (always-no accuracy: 0.987 )

Baseline (always guess 'not habitable') accuracy: 0.9868

AdaBoost, default settings (50 rounds, learning rate 1.0): accuracy 0.9894, recall 0.700, precision 0.583
[[1485, 10], [6, 14]]

AdaBoost, tuned (58 rounds, learning rate 1.0): accuracy 0.9894, recall 0.700, precision 0.583
[[1485, 10], [6, 14]]

AdaBoost, tuned settings, measurements only (no one-hot columns): accuracy 0.9894, recall 0.700, precision 0.583
[[1

In [22]:
# Downloading my files!
import shutil
shutil.make_archive('charts', 'zip', 'charts')
from google.colab import files
files.download('charts.zip')
files.download('exoplanets_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>